# La Copa del Decano · Laboratorio causal de 120 minutos (con huecos)

**Datos 100 % SINTÉTICOS.** La nota está en **0–50**, `pga_previo` en **0–5** y la fatiga latente `U` nunca se entrega a los participantes. Nada de aquí describe a personas reales.

## Cómo se trabaja este cuaderno
1. Sube a Colab (panel de archivos) los CSV que descargaste del juego: `sistema_academico_rector.csv`, `encuesta_estudiantes_juego.csv` y `cohorte_ab_test_2026.csv`. Si tu sala tiene menos de 4 estudiantes, las partes A–C usan una **referencia didáctica sintética rotulada** (no inventamos registros de tu sala).
2. Ejecuta la celda de preparación. Después busca las líneas que terminan en `None  # ⟵ HUECO …` y **reemplaza `None` por tu expresión**.
3. Cada hueco va seguido de `verificar(...)`, un **autocorrector** que compara tu resultado con un cálculo independiente y una **tolerancia numérica**. Si falla, verás una pista, no la solución. Si pasa, verás ✅.
4. Al final, el **tablero** resume qué huecos resolviste. No hay puntos, ranking ni cronómetro: este cuaderno es formativo y no se envía a ningún servidor.

## Ruta de Kolb en 120 minutos
| Fase | Dónde ocurre | Tu tarea |
|---|---|---|
| Experiencia concreta | El juego (15 exámenes) | Ya la viviste |
| Observación reflexiva | **Partes A y B** | Ver qué dicen las asociaciones y un Random Forest |
| Conceptualización abstracta | **Parte C** | Elegir el ajuste con el DAG: puerta trasera vs mal control |
| Experimentación activa | **Partes D y E** | A/B por estudiante y contraste con tu predicción |

> **Regla de oro:** un modelo que predice bien aprende `E[Y | X]`; una intervención pregunta por `E[Y | do(B)]`. Ni la importancia por permutación, ni SHAP, ni un R² alto cierran esa distancia.

In [ ]:
# ===== Preparación · no edites esta celda =====
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
try:
    from IPython.display import display
except ImportError:  # fuera de Jupyter
    display = print

RUTA = Path(".")   # carpeta de los CSV (en Colab: el panel de archivos)
TOL = 1e-6         # tolerancia numérica de los autocorrectores
RESULTADOS = {}
HUECOS = ["a1", "a2", "a3", "b1", "b2", "b3", "b4", "b5", "b6", "b7", "c1", "c2", "c3", "c4", "c5",
          "d1", "d2", "d3", "d4", "d5", "d6", "d7", "e1", "e2"]

def verificar(clave, valor, referencia=None, tol=TOL, condicion=None, pista=""):
    """Autocorrector: falla con una pista si el hueco sigue en None o si el cálculo no coincide."""
    assert valor is not None, f"❌ {clave}: completa el hueco (sigue en None)."
    if condicion is not None:
        assert condicion(valor), f"❌ {clave}: {pista}"
    if referencia is not None:
        assert np.allclose(np.asarray(valor, dtype=float), np.asarray(referencia, dtype=float), atol=tol, rtol=tol), (
            f"❌ {clave}: tu valor {np.round(np.asarray(valor, dtype=float), 4)} no coincide con el cálculo independiente "
            f"{np.round(np.asarray(referencia, dtype=float), 4)} (tolerancia {tol}). {pista}")
    RESULTADOS[clave] = True
    print(f"✅ {clave}")

def tablero():
    hechos = [h for h in HUECOS if RESULTADOS.get(h)]
    print(f"Huecos resueltos: {len(hechos)} de {len(HUECOS)}")
    for h in HUECOS:
        print(("✅" if RESULTADOS.get(h) else "⬜"), h)

# --- Referencia didáctica (misma estructura del DGP de la Copa; solo se usa si tu sala tiene < 4 estudiantes) ---
DIFICULTAD = [9.0, 8.5, 8.0, 7.5, 7.0, 6.0, 4.5, 4.0, 3.5, 3.0, 2.5]
# (estudiantes, rango PGA, (min, defecto, max) de estudio, sueño y bienestar) por arquetipo canónico
PERFILES = {
    "Sobreexigido": (8, (3.8, 4.8), (8, 11.5, 14), (2, 3.5, 5), (0, 0.6, 1.5)),
    "Equilibrado": (8, (3.4, 4.3), (4, 6, 8), (6.5, 7.5, 8.5), (1.5, 2.5, 3.5)),
    "En riesgo": (7, (2.0, 3.0), (2, 3.5, 5.5), (4.5, 6, 7), (1.5, 2.5, 4)),
    "Pragmático": (7, (3.0, 3.6), (0.5, 1.5, 3.5), (7, 8.5, 9.5), (3.5, 4.8, 6)),
}

def nota_dgp(pga, comp, est, sue, bie, ruido):
    # Misma estructura que el simulador de la Copa (fatiga latente U incluida); solo se usa en la referencia.
    bie = np.minimum(bie, 6.0)
    choque = np.where((sue < 4.0) & (est > 8.0), 15.0, 0.0)
    u = np.clip(10.0 + 60.0 / (1.0 + np.exp((sue - 4.0) * 1.2)) + 2.0 * np.maximum(0.0, est - 3.0) + choque
                + 1.5 * np.maximum(0.0, comp - 5.0) - 4.5 * bie + 2.0 * ruido, 0.0, 100.0)
    castigo = 0.15 * np.maximum(0.0, u - 20.0) + 0.12 * np.maximum(0.0, u - 60.0) ** 1.25
    return np.clip(30.0 + 10.0 * (pga - 3.5) - 1.8 * (comp - 5.5) + 2.6 * (est - 4.0) + 0.8 * (sue - 7.5)
                   + 0.5 * bie - castigo + 1.6 * ruido, 0.0, 50.0)

def referencia_didactica():
    # 30 estudiantes SINTÉTICOS × 15 exámenes con semilla fija. No son participantes ni tu sala.
    rng = np.random.default_rng(20261015)
    filas = []
    k = 0
    for arquetipo, (cantidad, (pga_lo, pga_hi), estudio, sueno, bienestar) in PERFILES.items():
        for _ in range(cantidad):
            k += 1
            pga = round(float(rng.uniform(pga_lo, pga_hi)), 1)
            for examen in range(15):
                elige = lambda t: float(np.clip(np.round(rng.triangular(t[0], t[1], t[2]) * 2) / 2, t[0], t[2]))
                est, sue, bie = elige(estudio), elige(sueno), elige(bienestar)
                filas.append(dict(estudiante_id="ref_%02d" % k, examen_nro=examen + 1, arquetipo=arquetipo,
                                  pga_previo=pga, complejidad=DIFICULTAD[examen % 11], horas_estudio=est,
                                  horas_sueno=sue, horas_bienestar_sem=bie, ruido=float(rng.normal())))
    d = pd.DataFrame(filas)
    d["nota_final"] = np.round(nota_dgp(d.pga_previo.to_numpy(), d.complejidad.to_numpy(), d.horas_estudio.to_numpy(),
                                        d.horas_sueno.to_numpy(), d.horas_bienestar_sem.to_numpy(), d.ruido.to_numpy()), 1)
    return d.drop(columns=["ruido"])

# --- Carga de datos ---
def leer(nombre):
    ruta = RUTA / nombre
    return pd.read_csv(ruta) if ruta.exists() else None

rector = leer("sistema_academico_rector.csv")
encuesta = leer("encuesta_estudiantes_juego.csv")
ab = leer("cohorte_ab_test_2026.csv")
d = None
if rector is not None and encuesta is not None:
    d = rector.merge(encuesta, on=["estudiante_id", "examen_nro"], validate="one_to_one")
if d is None or d.estudiante_id.nunique() < 4 or d.horas_bienestar_sem.nunique() < 2:
    d = referencia_didactica()
    ORIGEN = "REFERENCIA DIDÁCTICA INDEPENDIENTE · 30 estudiantes sintéticos · NO es tu sala"
else:
    ORIGEN = "TU SALA · datos sintéticos de la Copa"
print("Origen de los datos de las partes A–C:", ORIGEN)
print(len(d), "exámenes ·", d.estudiante_id.nunique(), "estudiantes")
print("Experimento A/B:", "no encontrado (descárgalo en la fase causal)" if ab is None else f"{len(ab)} filas · {ab.estudiante_id.nunique()} estudiantes sintéticos")

def coef_b(columnas, datos=None):
    """Coeficiente OLS de la PRIMERA columna (siempre B) en nota_final ~ columnas, con intercepto."""
    datos = d if datos is None else datos
    x = np.column_stack([np.ones(len(datos)), datos[list(columnas)].to_numpy(dtype=float)])
    return float(np.linalg.lstsq(x, datos.nota_final.to_numpy(dtype=float), rcond=None)[0][1])

## Parte A · Observación reflexiva: tres asociaciones

El Rector ve `H` (PGA previo), `Z` (complejidad), `B` (bienestar) y `Y` (nota). Calcula tres **coeficientes observacionales** de `B` sobre `Y` y mira cómo cambian al añadir columnas. Ninguno, por sí solo, es `do(B)`.

*Pista de orden:* en `coef_b([...])` la **primera** columna debe ser siempre `"horas_bienestar_sem"`.

In [ ]:
# A1 · pendiente CRUDA de nota_final sobre horas_bienestar_sem (una sola variable)
b_crudo = None  # ⟵ HUECO a1: np.polyfit(x, y, 1) devuelve [pendiente, intercepto]
verificar("a1", b_crudo, d.horas_bienestar_sem.cov(d.nota_final) / d.horas_bienestar_sem.var(),
          pista="Pendiente = cov(B, Y) / var(B). np.polyfit(x, y, 1) devuelve [pendiente, intercepto].")

# A2 · coeficiente de B ajustando SOLO por causas PREVIAS: PGA (H) y complejidad (Z)
b_antecedentes = None  # ⟵ HUECO a2: coef_b([...]) con B primero y luego las causas previas
verificar("a2", b_antecedentes,
          LinearRegression().fit(d[["horas_bienestar_sem", "pga_previo", "complejidad"]], d.nota_final).coef_[0],
          pista="Usa B + las causas PREVIAS. Estudio y sueño ocurren DESPUÉS de B.")

# A3 · coeficiente de B añadiendo además los mediadores POSTERIORES: estudio (X1) y sueño (S)
b_mediadores = None  # ⟵ HUECO a3: coef_b([...]) con B primero y todas las demás columnas del modelo ampliado
verificar("a3", b_mediadores,
          LinearRegression().fit(d[["horas_bienestar_sem", "pga_previo", "complejidad", "horas_estudio", "horas_sueno"]], d.nota_final).coef_[0],
          pista="Incluye las cinco columnas, con B en primer lugar.")

print(f"Crudo {b_crudo:+.2f} · con antecedentes {b_antecedentes:+.2f} · con mediadores {b_mediadores:+.2f} (puntos por hora de bienestar)")
plt.scatter(d.horas_bienestar_sem, d.nota_final, alpha=.35)
plt.xlabel("Horas semanales de bienestar B"); plt.ylabel("Nota Y (0–50)")
plt.title("Asociación observada (" + ORIGEN.split(" · ")[0] + ")"); plt.show()

**Reflexiona (sin respuesta única):** ¿por qué cambia el coeficiente de un modelo a otro? ¿Cuál de los tres responde a *«¿qué pasa si el programa impone 4 h?»*? Anota tu hipótesis; la Parte C la pondrá a prueba con el DAG.

## Parte B · Un Random Forest que *simula* do(B)

La segunda trampa: *«entreno un modelo flexible con todo y miro la importancia / SHAP»*. Aquí lo harás con **scikit-learn**: entrenas, mides la importancia por **permutación**, y **simulas la intervención** prediciendo las mismas filas con `B = 0` h y con `B = 4` h. Es la misma simulación que ejecutaste en el juego.

In [ ]:
covariables_rector = ["pga_previo", "complejidad", "horas_bienestar_sem"]
X, y = d[covariables_rector], d.nota_final
entrenar, prueba = next(GroupShuffleSplit(n_splits=1, test_size=.33, random_state=19).split(X, y, groups=d.estudiante_id))
print("Entrenamiento:", len(entrenar), "filas · prueba:", len(prueba), "filas (split por estudiante: nadie aparece en ambos)")

# B1 · entrena el bosque SOLO con las filas de entrenamiento
bosque = None  # ⟵ HUECO b1: RandomForestRegressor con 45 árboles, min_samples_leaf=3, random_state=13, n_jobs=1; ajústalo (.fit) solo con entrenamiento
referencia_bosque = RandomForestRegressor(n_estimators=45, min_samples_leaf=3, random_state=13, n_jobs=1)
referencia_bosque.fit(X.iloc[entrenar], y.iloc[entrenar])
verificar("b1", None if bosque is None else bosque.predict(X.iloc[prueba]), referencia_bosque.predict(X.iloc[prueba]),
          pista="Mismos hiperparámetros y SOLO filas de entrenamiento (X.iloc[entrenar]); el resultado es determinista con random_state=13.")

# B2 · R² del bosque en las filas de PRUEBA
r2_prueba = None  # ⟵ HUECO b2: r2_score(y_verdadero, y_predicho) con las filas de prueba
verificar("b2", r2_prueba, bosque.score(X.iloc[prueba], y.iloc[prueba]), pista="R² del bosque sobre las filas de PRUEBA, no de entrenamiento.")
print("MAE de prueba:", round(mean_absolute_error(y.iloc[prueba], bosque.predict(X.iloc[prueba])), 2), "puntos · R² de prueba:", round(r2_prueba, 3))

# B3 · importancia por PERMUTACIÓN en la prueba (cuánto R² se pierde al barajar cada columna)
importancia = None  # ⟵ HUECO b3: permutation_importance con n_repeats=8, random_state=7 y scoring="r2", sobre la prueba
def importancia_manual(modelo, Xp, yp, repeticiones=8, semilla=99):
    """Ruta independiente del autocorrector: barajar a mano cada columna y medir cuánto cae el R²."""
    azar, base, medias = np.random.default_rng(semilla), modelo.score(Xp, yp), []
    for j in range(Xp.shape[1]):
        caidas = []
        for _ in range(repeticiones):
            barajada = Xp.copy()
            barajada.iloc[:, j] = azar.permutation(barajada.iloc[:, j].to_numpy())
            caidas.append(base - modelo.score(barajada, yp))
        medias.append(np.mean(caidas))
    return np.array(medias)
verificar("b3", None if importancia is None else importancia.importances_mean, importancia_manual(bosque, X.iloc[prueba], y.iloc[prueba]), tol=0.3,
          condicion=lambda _: importancia.importances.shape == (len(covariables_rector), 8),
          pista="Tu objeto debe tener 8 repeticiones por columna (n_repeats=8) y salir de la PRUEBA; los barajados al azar difieren un poco, por eso la tolerancia es 0.3.")
for i in np.argsort(-importancia.importances_mean):
    print(f"  {covariables_rector[i]:<22} {importancia.importances_mean[i]:6.3f}")
print("La importancia mide utilidad PREDICTIVA, no el efecto de intervenir esa variable.")

In [ ]:
# El bosque que simula la intervención se reentrena con TODAS las filas (evaluar honestamente → reentrenar → simular)
bosque_final = RandomForestRegressor(n_estimators=45, min_samples_leaf=3, random_state=13, n_jobs=1).fit(X, y)

# B4 · las MISMAS filas, pero con B = 0 h para todas
X_bajo = None  # ⟵ HUECO b4: X.assign(columna=valor) devuelve una copia con esa columna fija
referencia_bajo = X.copy(); referencia_bajo["horas_bienestar_sem"] = 0.0
verificar("b4", None if X_bajo is None else X_bajo.to_numpy(dtype=float), referencia_bajo.to_numpy(dtype=float),
          pista="Fija B en 0 horas y deja el resto de columnas como se observaron.")

# B5 · las MISMAS filas, pero con B = 4 h para todas (el programa institucional)
X_alto = None  # ⟵ HUECO b5: igual que b4, con el valor del programa institucional
referencia_alto = X.copy(); referencia_alto["horas_bienestar_sem"] = 4.0
verificar("b5", None if X_alto is None else X_alto.to_numpy(dtype=float), referencia_alto.to_numpy(dtype=float),
          pista="Fija B en 4 horas y deja el resto de columnas como se observaron.")

# B6 · efecto SIMULADO = media de (predicción con B=4) − (predicción con B=0)
efecto_sim = None  # ⟵ HUECO b6: promedia la diferencia de predicciones de bosque_final entre ambos futuros
verificar("b6", efecto_sim, float(np.mean(bosque_final.predict(referencia_alto) - bosque_final.predict(referencia_bajo))),
          pista="Predice X_alto y X_bajo con bosque_final y promedia la diferencia.")

# B7 · dependencia parcial de B: nota prevista media si TODAS las filas tuvieran ese B
rejilla = [0, 1, 2, 3, 4, 5, 6]
pdp = None  # ⟵ HUECO b7: una lista con una nota prevista media por cada g de la rejilla
verificar("b7", pdp, [bosque_final.predict(referencia_alto.assign(horas_bienestar_sem=float(g))).mean() for g in rejilla],
          condicion=lambda v: abs((v[4] - v[0]) - efecto_sim) < TOL,
          pista="Una media de predicciones por cada g de la rejilla; además pdp[4] - pdp[0] debe coincidir con tu efecto_sim (es la misma simulación).")
print(f"Efecto simulado do(B=4) − do(B=0): {efecto_sim:+.2f} puntos")
plt.plot(rejilla, pdp, marker="o"); plt.xlabel("B impuesto (h/sem)"); plt.ylabel("Nota prevista media")
plt.title("Dependencia parcial de B (simulación predictiva, NO efecto causal)"); plt.show()
bien = d.horas_bienestar_sem
print(f"Cobertura: solo {100 * (bien <= 1).mean():.0f} % de los exámenes con B ≤ 1 h y {100 * (bien >= 3).mean():.0f} % con B ≥ 3 h. "
      "Fuera de lo observado el bosque repite el valor del borde: es extrapolación.")

In [ ]:
# Dado: repite la simulación con un modelo "más completo" que añade estudio (X1) y sueño (S), fijados en lo observado.
def simular_do_b(columnas, semilla=13):
    Xc = d[columnas]
    m = RandomForestRegressor(n_estimators=45, min_samples_leaf=3, random_state=semilla, n_jobs=1).fit(Xc, d.nota_final)
    return float((m.predict(Xc.assign(horas_bienestar_sem=4.0)) - m.predict(Xc.assign(horas_bienestar_sem=0.0))).mean())

covariables_encuesta = covariables_rector + ["horas_estudio", "horas_sueno"]
efecto_sim_5 = simular_do_b(covariables_encuesta)
Xe, ye = d[covariables_encuesta], d.nota_final
r2_5 = RandomForestRegressor(n_estimators=45, min_samples_leaf=3, random_state=13, n_jobs=1).fit(Xe.iloc[entrenar], ye.iloc[entrenar]).score(Xe.iloc[prueba], ye.iloc[prueba])
print(f"R² de prueba: {r2_prueba:.3f} (3 variables) → {r2_5:.3f} (5 variables)")
print(f"Efecto simulado de B: {efecto_sim:+.2f} → {efecto_sim_5:+.2f}")
print("Con X1 y S fijos, la simulación solo conserva el efecto de B que NO pasa por esos mediadores.")

**Reflexiona:** la importancia de `B` puede ser casi cero mientras la simulación da otro número, y viceversa. ¿Qué lectura te tentaría a *recortar* el presupuesto? ¿Cuál a *ampliarlo*? ¿Qué te falta para decidir con seguridad? (Esto es lo que anotaste en el juego.)

## Parte C · Conceptualización: puerta trasera vs mal control

Las flechas del DAG son **supuestos del simulador**, no hallazgos de la correlación. Con ellas se decide qué ajustar para el efecto **total** de `B` sobre `Y`:

* **Puerta trasera:** ajusta las causas *previas* que abren caminos `B ← H → Y`.
* **Mal control:** ajustar un *descendiente* de `B` (mediador) bloquea parte del efecto total, y la fatiga `U` ni siquiera se observa.

In [ ]:
DAG = {  # causa -> efectos directos (supuestos del DGP de la Copa)
    "H": ["B", "X1", "Y"], "Z": ["X1", "U", "Y"], "B": ["X1", "S", "U", "Y"],
    "X1": ["S", "U", "Y"], "S": ["U", "Y"], "U": ["Y"], "Y": []}
COLUMNA = {"H": "pga_previo", "Z": "complejidad", "B": "horas_bienestar_sem", "X1": "horas_estudio", "S": "horas_sueno"}  # U no se observa

def descendientes(grafo, nodo):
    vistos, pendientes = set(), list(grafo[nodo])
    while pendientes:
        actual = pendientes.pop()
        if actual not in vistos:
            vistos.add(actual); pendientes.extend(grafo[actual])
    return vistos

def padres(grafo, nodo):
    return {causa for causa, efectos in grafo.items() if nodo in efectos}

# C1 · ¿qué variables son DESCENDIENTES de B (posteriores a la intervención)?
desc_b = None  # ⟵ HUECO c1: hay una función descendientes(grafo, nodo) más arriba
verificar("c1", desc_b, condicion=lambda v: set(v) == {"X1", "S", "U", "Y"},
          pista="Descendientes = todo lo que B alcanza siguiendo las flechas hacia adelante.")

# C2 · elige columnas OBSERVADAS para cerrar la puerta trasera de B (sin descendientes de B)
ajuste = None  # ⟵ HUECO c2: lista de nombres de columna; Z es opcional (mejora precisión), H no
verificar("c2", ajuste, condicion=lambda v: COLUMNA["H"] in v and not (set(v) & {COLUMNA[n] for n in desc_b if n in COLUMNA}) and set(v) <= {"pga_previo", "complejidad"},
          pista="Debe incluir la causa previa de B (padres de B en el DAG) y NINGÚN descendiente de B.")
print("Padres de B en el DAG:", sorted(padres(DAG, "B")), "· ajuste elegido:", ajuste)

# C3 · coeficiente de B con el ajuste de puerta trasera
b_backdoor = None  # ⟵ HUECO c3: coef_b con B primero y después tu ajuste
verificar("c3", b_backdoor, LinearRegression().fit(d[["horas_bienestar_sem"] + list(ajuste)], d.nota_final).coef_[0],
          pista="B primero y después tu conjunto de ajuste.")

# C4 · el MAL CONTROL: el mismo ajuste más los mediadores X1 y S
b_mal_control = None  # ⟵ HUECO c4: coef_b con B, tu ajuste y además los mediadores
verificar("c4", b_mal_control,
          LinearRegression().fit(d[["horas_bienestar_sem"] + list(ajuste) + ["horas_estudio", "horas_sueno"]], d.nota_final).coef_[0],
          pista="Añade horas_estudio y horas_sueno a tu ajuste.")
print(f"Puerta trasera {b_backdoor:+.2f} · mal control {b_mal_control:+.2f} · diferencia {b_mal_control - b_backdoor:+.2f} pts/h")

# Dado: remuestreo de ESTUDIANTES (no de exámenes) para ver la incertidumbre del coeficiente de puerta trasera
def bootstrap_coef(columnas, reps=300, semilla=2026):
    azar = np.random.default_rng(semilla)
    ids = d.estudiante_id.unique()
    grupos = {i: g for i, g in d.groupby("estudiante_id")}
    return np.array([coef_b(columnas, pd.concat([grupos[i] for i in azar.choice(ids, size=len(ids), replace=True)])) for _ in range(reps)])

remuestreo = bootstrap_coef(["horas_bienestar_sem"] + list(ajuste))
# C5 · intervalo bootstrap 95 % (percentiles 2,5 y 97,5)
ic_backdoor = None  # ⟵ HUECO c5: percentiles 2,5 y 97,5 del remuestreo
verificar("c5", ic_backdoor, np.quantile(remuestreo, [0.025, 0.975]), pista="Percentiles 2.5 y 97.5 del remuestreo (cuantiles 0.025 y 0.975).")
print(f"Puerta trasera {b_backdoor:+.2f} pts/h · IC95 bootstrap por estudiante [{ic_backdoor[0]:+.2f}, {ic_backdoor[1]:+.2f}]")

**Reflexiona:** ajustar por `H` (y `Z`) *puede* acercarte al efecto total **solo si** el DAG es correcto, no hay confusión no medida y B se observa en todo su rango (positividad). Nada de eso lo garantiza un dataset observacional; el experimento sí lo hace por diseño.

## Parte D · Experimentación activa: el A/B por estudiante

`cohorte_ab_test_2026.csv` tiene **30 estudiantes sintéticos × 15 exámenes**. `tratamiento = 1` impone **B = 4 h**; `0` impone **B = 0 h**. El sorteo ocurrió **por estudiante**, así que la unidad de análisis es el estudiante, no el examen.

In [ ]:
assert ab is not None, "Descarga cohorte_ab_test_2026.csv cuando el docente libere la fase causal y súbelo a Colab."

# D1 · promedia la nota POR ESTUDIANTE conservando su brazo de tratamiento
por_estudiante = None  # ⟵ HUECO d1: agrupa por estudiante Y tratamiento y promedia nota_final (as_index=False)
verificar("d1", None if por_estudiante is None else por_estudiante.nota_final.mean(), ab.groupby("estudiante_id").nota_final.mean().mean(),
          condicion=lambda _: len(por_estudiante) == ab.estudiante_id.nunique() and {"estudiante_id", "tratamiento", "nota_final"} <= set(por_estudiante.columns),
          pista="Una fila por estudiante (30), no una por examen (450), que conserve estudiante_id, tratamiento y el promedio de nota_final.")

# D2 · notas promedio de los TRATADOS (B = 4 h)
tratados = None  # ⟵ HUECO d2: filtra por_estudiante por brazo y quédate con nota_final
verificar("d2", None if tratados is None else sorted(tratados.tolist()), sorted(ab[ab.tratamiento == 1].groupby("estudiante_id").nota_final.mean().tolist()),
          pista="Solo estudiantes con tratamiento == 1.")

# D3 · notas promedio de los CONTROLES (B = 0 h)
controles = None  # ⟵ HUECO d3: igual que d2 para el otro brazo
verificar("d3", None if controles is None else sorted(controles.tolist()), sorted(ab[ab.tratamiento == 0].groupby("estudiante_id").nota_final.mean().tolist()),
          pista="Solo estudiantes con tratamiento == 0.")

# D4 · ATE observado = media de tratados − media de controles
ate = None  # ⟵ HUECO d4
verificar("d4", ate, ab[ab.tratamiento == 1].groupby("estudiante_id").nota_final.mean().mean() - ab[ab.tratamiento == 0].groupby("estudiante_id").nota_final.mean().mean(),
          pista="Diferencia de medias entre estudiantes.")

# D5 · error estándar entre estudiantes (varianza muestral, ddof=1, en cada brazo)
se = None  # ⟵ HUECO d5: raíz de la suma de varianza muestral / n en cada brazo
verificar("d5", se, abs(ate / stats.ttest_ind(tratados, controles, equal_var=False).statistic),
          pista="Welch: se = sqrt(var(t, ddof=1)/n_t + var(c, ddof=1)/n_c).")

# D6 · IC 95 % normal aproximado
ic95 = None  # ⟵ HUECO d6: una tupla (inferior, superior) alrededor del ATE
verificar("d6", ic95, ate + np.array([-1.96, 1.96]) * se, pista="El IC95 normal aproximado es ate ± 1.96 × se.")
print(f"ATE observado {ate:+.2f} puntos · IC95 entre estudiantes [{ic95[0]:+.2f}, {ic95[1]:+.2f}] · {len(tratados)} tratados / {len(controles)} controles")

# D7 · el error ingenuo que trata los 450 exámenes como 450 personas independientes
n1, n0 = int((ab.tratamiento == 1).sum()), int((ab.tratamiento == 0).sum())
se_ingenuo = None  # ⟵ HUECO d7: el mismo cálculo que d5, pero con las notas de TODAS las filas (n1 y n0 ya están definidos)
verificar("d7", se_ingenuo, np.sqrt(ab.loc[ab.tratamiento == 1, "nota_final"].var(ddof=1) / n1 + ab.loc[ab.tratamiento == 0, "nota_final"].var(ddof=1) / n0),
          condicion=lambda v: v < se,
          pista="Mismo cálculo que d5, pero con las notas de cada examen; debe salir MENOR que el error correcto (falsa precisión).")
print(f"Error ingenuo {se_ingenuo:.2f} < error correcto {se:.2f}: tratar exámenes como personas finge una precisión que no existe.")
print("Balance basal de PGA por brazo:", ab.groupby("tratamiento").pga_previo.mean().round(2).to_dict())

## Parte E · Contraste: tu predicción frente al experimento

Pon todas las estimaciones en la misma escala y pregunta: **¿cuáles caen dentro del IC95 del experimento?** Coincidir una vez no demuestra que un método identifique el efecto; el A/B lo garantiza por diseño y, aun así, su IC puede ser amplio con solo 30 estudiantes.

In [ ]:
estimaciones = {
    "OLS cruda (A1)": b_crudo, "OLS con antecedentes (A2)": b_antecedentes, "OLS con mediadores (A3)": b_mediadores,
    "Puerta trasera (C3)": b_backdoor, "Mal control (C4)": b_mal_control,
    "Random Forest do(B), 3 variables (B6)": efecto_sim, "Random Forest do(B), 5 variables": efecto_sim_5}

# E1 · para cada estimación: ¿cae dentro del IC95 del A/B?
dentro_ic = None  # ⟵ HUECO e1: un diccionario {nombre: ¿está el valor dentro del IC95?}
verificar("e1", None if dentro_ic is None else [dentro_ic[k] for k in estimaciones], [ic95[0] <= v <= ic95[1] for v in estimaciones.values()],
          pista="Un booleano por estimación, comparando con el IC95 del experimento.")
resumen = pd.DataFrame({"estimación": list(estimaciones), "puntos": [round(v, 2) for v in estimaciones.values()], "dentro del IC95 del A/B": list(dentro_ic.values())})
display(resumen)
print(f"A/B: ATE {ate:+.2f} · IC95 [{ic95[0]:+.2f}, {ic95[1]:+.2f}] (30 estudiantes sintéticos; la verdad sintética solo la conoce el docente)")

# E2 · tu recomendación final al Rector, DESPUÉS de ver todo
mi_recomendacion = None  # ⟵ HUECO e2: "recortar", "mantener", "ampliar" o "experimentar"
verificar("e2", mi_recomendacion, condicion=lambda v: v in {"recortar", "mantener", "ampliar", "experimentar"},
          pista="Elige una de las cuatro opciones entre comillas. No hay respuesta oculta: se discute en plenaria.")
print("Tu recomendación:", mi_recomendacion, "→ compárala con la que anotaste en el juego y justifica el cambio (o no) con evidencia.")

## Tablero y lista de verificación

In [ ]:
tablero()

- ¿Distingues la asociación del Rector de `do(B)`, y la importancia predictiva del efecto de intervenir?
- ¿Sabes por qué `X1` y `S` (descendientes de `B`) son *malos controles* para el efecto total?
- ¿Tu simulación del Random Forest quedó dentro o fuera del IC95 del A/B, y qué dice eso (y qué NO) sobre el método?
- ¿La asignación A/B fue por estudiante y tu intervalo usa estudiantes, no 450 exámenes?
- ¿Tus recomendaciones comunican incertidumbre, cupos y límites de un mundo **sintético**?

Para el docente existe un CSV adicional de resultados potenciales y fatiga sintética. **No se entrega a los participantes antes del cierre conceptual.** EconML del juego V2 anterior corresponde a otro DGP y no se compara directamente con estas notas.